In [ ]:
#pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.2 MB/s eta 0:00:00


In [ ]:

import optuna
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
files = ["46343_30_dataset.csv", "759667_30_dataset.csv",
         "781756_30_dataset.csv", "844359_30_dataset.csv",
         "1066528_30_dataset.csv", "1360686_30_dataset.csv",
         "1449548_30_dataset.csv", "1455390_30_dataset.csv",
         "1818471_30_dataset.csv", "2598705_30_dataset.csv",
         "2638030_30_dataset.csv", "3509524_30_dataset.csv",
         "3997827_30_dataset.csv", "4018081_30_dataset.csv",
         "4314139_30_dataset.csv", "4426783_30_dataset.csv",
         "5132496_30_dataset.csv", "5383425_30_dataset.csv",
         "5498603_30_dataset.csv", "5797046_30_dataset.csv",
         "6220552_30_dataset.csv", "7749105_30_dataset.csv",
         "8000685_30_dataset.csv", "8173033_30_dataset.csv",
         "8258170_30_dataset.csv", "8530312_30_dataset.csv",
         "8686948_30_dataset.csv", "8692923_30_dataset.csv",
         "9106476_30_dataset.csv", "9961348_30_dataset.csv"]

test_files = ["9618981_30_dataset.csv"]

In [ ]:
from pathlib import Path

In [ ]:
df_list = []
for file in files:
    temp_df = pd.read_csv(file)

    file_name = Path(file).name

    try:
        # Розбиваємо назву файлу за нижнім підкресленням
        user_id = file_name.split('_')[0]
        temp_df['user_id'] = user_id
    except Exception as e:
        print(f"Помилка обробки файлу {file}: {e}")
        continue

    df_list.append(temp_df)

combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
test_df = pd.read_csv(test_files[0])
test_df['user_id'] = "9618981"

In [ ]:

features = ['rr_norm_diff', 'rr_delta', 'mean_enmo', 'max_enmo', 'std_enmo']
label = "sleep_stage"

# 2. ЖОРСТКИЙ МАПІНГ КЛАСІВ (Зведення до 4 бізнес-фаз для будильника)
mapping_dict = {
    0: 0, # Wake
    1: 1, # N1 -> Light Sleep
    2: 1, # N2 -> Light Sleep
    3: 2, # N3 -> Deep Sleep
    4: 3, # REM
    5: 3  # На всякий випадок, якщо десь лишилися оригінальні мітки
}

In [ ]:

# Застосовуємо мапінг
combined_df[label] = combined_df[label].replace(mapping_dict)
test_df[label] = test_df[label].replace(mapping_dict)

# Відкидаємо артефакти (-1), які могли залишитися
combined_df = combined_df[combined_df[label] >= 0]
test_df = test_df[test_df[label] >= 0]

# Формуємо датасети
X_train_rf = combined_df[features]
Y_train_rf = combined_df[label]
groups_train = combined_df['user_id'].values # Для коректної крос-валідації

X_test_rf = test_df[features]
Y_test_rf = test_df[label]

In [ ]:
# 3. ФУНКЦІЯ ДЛЯ OPTUNA (Байєсівський пошук від перенавчання)
def objective(trial):
    rf_params = {
        # Скільки дерев у лісі
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        # Жорстке обмеження глибини дерева (лікуємо перенавчання!)
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        # Скільки епох мінімум має бути в "листку", щоб прийняти рішення
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 25),
        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42
    }

    model = RandomForestClassifier(**rf_params)

    # Крос-валідація (4 фолди). GroupKFold гарантує,
    # що епохи одного юзера не потраплять одночасно в train і validation!
    gkf = GroupKFold(n_splits=4)
    scores = cross_val_score(model, X_train_rf, Y_train_rf, cv=gkf, groups=groups_train, scoring='f1_macro')

    return scores.mean()

In [ ]:
print("Запускаємо Байєсівський пошук Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\nНайкращі знайдені параметри (Optuna):")
print(study.best_params)

[I 2026-04-17 10:09:39,622] A new study created in memory with name: no-name-a1a40c70-68a1-47bd-9b5f-89e2854e804f


Запускаємо Байєсівський пошук Optuna...


[I 2026-04-17 10:10:19,775] Trial 0 finished with value: 0.407758591935957 and parameters: {'n_estimators': 288, 'max_depth': 8, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.407758591935957.
[I 2026-04-17 10:10:43,263] Trial 1 finished with value: 0.3857495376992768 and parameters: {'n_estimators': 276, 'max_depth': 5, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.407758591935957.
[I 2026-04-17 10:11:04,390] Trial 2 finished with value: 0.37721023876067106 and parameters: {'n_estimators': 286, 'max_depth': 4, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.407758591935957.
[I 2026-04-17 10:11:29,802] Trial 3 finished with value: 0.3930693021380736 and parameters: {'n_estimators': 248, 'max_depth': 6, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.407758591935957.
[I 2026-04-17 10:11:46,762] Trial 4 finished with value: 0.3763057165533913 and parameters: {'n_estimators': 234, 'max_depth': 4, 'min_samples_leaf': 17}. Best is trial 0 with value: 0.40775859


Найкращі знайдені параметри (Optuna):
{'n_estimators': 86, 'max_depth': 10, 'min_samples_leaf': 22}


In [ ]:
# 4. ТРЕНУВАННЯ ФІНАЛЬНОЇ МОДЕЛІ З НАЙКРАЩИМИ ПАРАМЕТРАМИ
print("\nТренуємо фінальний Random Forest...")
best_rf = RandomForestClassifier(**study.best_params, class_weight='balanced', n_jobs=-1, random_state=42)
best_rf.fit(X_train_rf, Y_train_rf)


# 5. ОЦІНКА РЕЗУЛЬТАТІВ
rf_preds = best_rf.predict(X_test_rf)
target_names = ['Wake', 'Light Sleep', 'Deep Sleep', 'REM']

print("\n--- Результати оптимізованого Random Forest ---")
print(f"Загальна точність (Accuracy): {accuracy_score(Y_test_rf, rf_preds):.3f}")
print(classification_report(Y_test_rf, rf_preds, target_names=target_names))


Тренуємо фінальний Random Forest...

--- Результати оптимізованого Random Forest ---
Загальна точність (Accuracy): 0.343
              precision    recall  f1-score   support

        Wake       0.51      0.43      0.46        94
 Light Sleep       0.63      0.30      0.41       513
  Deep Sleep       0.02      0.20      0.03        25
         REM       0.42      0.39      0.41       313

    accuracy                           0.34       945
   macro avg       0.39      0.33      0.33       945
weighted avg       0.53      0.34      0.40       945



In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(Y_test_rf, rf_preds)

array([[ 37,  25,  10,  22],
       [ 18, 151, 212, 132],
       [  1,   0,   5,  19],
       [ 17,  69, 106, 121]])